# Cross-Country Climate Comparison & Vulnerability Ranking

Analysis comparing climate data across Ethiopia, Kenya, Sudan, Tanzania, and Nigeria to inform COP32 positioning.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# Load and Concatenate Cleaned Datasets
countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']
dfs = []

for country in countries:
    df = pd.read_csv(f'data/{country}_clean.csv')
    dfs.append(df)

# Concatenate all datasets
combined_df = pd.concat(dfs, ignore_index=True)

# Ensure Date is datetime
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Extract Year and Month if needed
combined_df['Year'] = combined_df['Date'].dt.year
combined_df['Month'] = combined_df['Date'].dt.month

print(f'Combined dataset shape: {combined_df.shape}')
print(f'Countries included: {combined_df["Country"].unique()}')
print(f'Date range: {combined_df["Date"].min()} to {combined_df["Date"].max()}')

## Temperature Trend Comparison

Comparing monthly average temperature (T2M) across all five countries from 2015-2026.

In [ ]:
# Monthly Average T2M for All Countries
monthly_temp = combined_df.groupby(['Country', 'Month'])['T2M'].mean().reset_index()

plt.figure(figsize=(14, 8))
for country in countries:
    country_data = monthly_temp[monthly_temp['Country'] == country.capitalize()]
    plt.plot(country_data['Month'], country_data['T2M'], marker='o', label=country.capitalize(), linewidth=2)

plt.title('Monthly Average Temperature (T2M) Comparison Across Countries (2015-2026)', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

In [ ]:
# Temperature Summary Statistics
temp_stats = combined_df.groupby('Country')['T2M'].agg(['mean', 'median', 'std']).round(2)
temp_stats.columns = ['Mean T2M (°C)', 'Median T2M (°C)', 'Std Dev T2M (°C)']
print('Temperature Statistics Across Countries:')
print(temp_stats)

## Precipitation Variability Comparison

Comparing precipitation variability across countries using boxplots and summary statistics.

In [ ]:
# Side-by-side Boxplots of PRECTOTCORR
plt.figure(figsize=(12, 8))
sns.boxplot(data=combined_df, x='Country', y='PRECTOTCORR', showfliers=False)
plt.title('Precipitation Variability Comparison Across Countries', fontsize=14)
plt.xlabel('Country')
plt.ylabel('Precipitation (mm/day)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precipitation Summary Statistics
precip_stats = combined_df.groupby('Country')['PRECTOTCORR'].agg(['mean', 'median', 'std']).round(2)
precip_stats.columns = ['Mean Precip (mm/day)', 'Median Precip (mm/day)', 'Std Dev Precip (mm/day)']
print('Precipitation Statistics Across Countries:')
print(precip_stats)

## Extreme Event Frequency

Analyzing extreme heat days and consecutive dry days per year.

In [ ]:
# Extreme Heat Days (T2M_MAX > 35°C)
extreme_heat = combined_df[combined_df['T2M_MAX'] > 35]
heat_days_per_year = extreme_heat.groupby(['Country', 'Year']).size().reset_index(name='Extreme Heat Days')

plt.figure(figsize=(12, 8))
sns.barplot(data=heat_days_per_year, x='Year', y='Extreme Heat Days', hue='Country')
plt.title('Extreme Heat Days per Year (T2M_MAX > 35°C)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Number of Days')
plt.legend(title='Country')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Consecutive Dry Days (PRECTOTCORR < 1 mm)
# Function to calculate max consecutive dry days per year per country
def max_consecutive_dry_days(group):
    dry_days = (group['PRECTOTCORR'] < 1).astype(int)
    max_consecutive = 0
    current = 0
    for day in dry_days:
        if day == 1:
            current += 1
            max_consecutive = max(max_consecutive, current)
        else:
            current = 0
    return max_consecutive

dry_days_per_year = combined_df.groupby(['Country', 'Year']).apply(max_consecutive_dry_days).reset_index(name='Max Consecutive Dry Days')

plt.figure(figsize=(12, 8))
sns.barplot(data=dry_days_per_year, x='Year', y='Max Consecutive Dry Days', hue='Country')
plt.title('Maximum Consecutive Dry Days per Year (PRECTOTCORR < 1 mm)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Number of Consecutive Days')
plt.legend(title='Country')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Statistical Testing

Testing for significant differences in temperature across countries.

In [ ]:
# One-way ANOVA on T2M across countries
country_groups = [group['T2M'].values for name, group in combined_df.groupby('Country')]

# Perform ANOVA
f_stat, p_value = stats.f_oneway(*country_groups)

print(f'ANOVA Results for T2M across countries:')
print(f'F-statistic: {f_stat:.4f}')
print(f'P-value: {p_value:.6f}')

if p_value < 0.05:
    print('Result: Significant differences in temperature means across countries (p < 0.05)')
else:
    print('Result: No significant differences in temperature means across countries (p >= 0.05)')

## Vulnerability Ranking

Ranking countries by climate vulnerability based on temperature trends, precipitation variability, and extreme events.

In [ ]:
# Vulnerability Scoring (higher score = higher vulnerability)
# Factors: temperature mean (higher = more vulnerable), precipitation std dev (higher = more variable = vulnerable), 
# extreme heat days average, max dry days average

vulnerability_df = pd.DataFrame({
    'Country': countries,
    'Avg Temperature': temp_stats['Mean T2M (°C)'],
    'Precip Variability': precip_stats['Std Dev Precip (mm/day)'],
    'Avg Extreme Heat Days': heat_days_per_year.groupby('Country')['Extreme Heat Days'].mean(),
    'Avg Max Dry Days': dry_days_per_year.groupby('Country')['Max Consecutive Dry Days'].mean()
})

# Normalize and score (higher values indicate higher vulnerability)
vulnerability_df['Temp Score'] = (vulnerability_df['Avg Temperature'] - vulnerability_df['Avg Temperature'].min()) / (vulnerability_df['Avg Temperature'].max() - vulnerability_df['Avg Temperature'].min())
vulnerability_df['Precip Score'] = (vulnerability_df['Precip Variability'] - vulnerability_df['Precip Variability'].min()) / (vulnerability_df['Precip Variability'].max() - vulnerability_df['Precip Variability'].min())
vulnerability_df['Heat Score'] = (vulnerability_df['Avg Extreme Heat Days'] - vulnerability_df['Avg Extreme Heat Days'].min()) / (vulnerability_df['Avg Extreme Heat Days'].max() - vulnerability_df['Avg Extreme Heat Days'].min())
vulnerability_df['Dry Score'] = (vulnerability_df['Avg Max Dry Days'] - vulnerability_df['Avg Max Dry Days'].min()) / (vulnerability_df['Avg Max Dry Days'].max() - vulnerability_df['Avg Max Dry Days'].min())

vulnerability_df['Total Vulnerability Score'] = vulnerability_df[['Temp Score', 'Precip Score', 'Heat Score', 'Dry Score']].mean(axis=1)

# Rank by total score
vulnerability_ranking = vulnerability_df.sort_values('Total Vulnerability Score', ascending=False)[['Country', 'Total Vulnerability Score']].reset_index(drop=True)
vulnerability_ranking['Rank'] = range(1, len(vulnerability_ranking) + 1)

print('Climate Vulnerability Ranking:')
print(vulnerability_ranking)

## COP32 Position Paper Framing

### Key Findings for Ethiopia's COP32 Strategy:

1. **Temperature Trends**: [Country with highest warming] is experiencing the fastest temperature increase, with an average T2M of [value]°C compared to [lowest] at [value]°C, indicating urgent need for adaptation measures in hotter regions.

2. **Precipitation Instability**: [Country with highest variability] shows the most unstable precipitation patterns with a standard deviation of [value] mm/day, suggesting greater vulnerability to droughts and floods.

3. **Extreme Events**: [Country with most extreme heat/dry days] faces the highest climate stress with an average of [heat days] extreme heat days and [dry days] consecutive dry days per year, highlighting the need for early warning systems.

4. **Ethiopia's Position**: Ethiopia ranks [position] in vulnerability with moderate temperature and precipitation patterns, positioning it well to lead regional climate cooperation while addressing its own adaptation needs.

5. **Priority for Climate Finance**: [Most vulnerable country] should be Ethiopia's champion at COP32 for priority climate finance, as the data shows [brief justification based on scores], supporting targeted investments in resilience building.